# Born effective charge: DFPT vs Wannier mixed curvature

Ti Born effective charge in cubic SrTiO3: compares the row `(Z*_{Ti,zx}, Z*_{Ti,zy}, Z*_{Ti,zz})` from Quantum ESPRESSO DFPT against a one-sided finite difference between the undisplaced structure and `Ti_Q_01A`, where Ti alone moves by `+0.01 A` along z.

**Method.** With `u = u_{Ti,z}` the electronic response is the BZ integral of the trace of the mixed `(k, u)` curvature:
$$Z^{el}_{Ti,z\beta} = \frac{\Omega}{(2\pi)^3} \int d^3k_{cart}\, \mathrm{Tr}\, B_{k_\beta, u}$$
`curvature.fields` and `TBModel.velocity` both work in **reduced** reciprocal axes, so the Cartesian prefactor is applied once, at the end, via `B_inv = (recip_lat_vecs)^-1`. The curvature is split as `B = B_int + B_cross + B_ext` and each piece is integrated separately -- only the sum is physical. Adding the ionic term to the zz component gives the total to compare with `ph.x`.

**The ionic term is not `z_valence`.** `dis_win_min` leaves the Ti 3s/3p semicore bands outside the Wannier space; they translate rigidly with the displaced atom, so `Z^ion,eff = z_valence - N_EXCLUDED_ON_DISPLACED` (Ti: 12 - 8 = +4). `Z*_{Ti,zz}` should come out near `Z*_QE - Z^ion,eff = 7.29 - 4 = +3.29`.

Set `RUN_COMPUTATION = False` to skip straight to loading the last saved `results` from this trial/flag combination's `born_charge/` directory (next to the Wannier90 run that produced it, not under `validation/results/` -- see the Config cell).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import csv

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "validation").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from validation._paths import STO
from validation._paths import DATA
from validation.observable.born_effective_charge_lib import (
    PREFIX,
    BASE_RUN,
    PH_OUT as DEFAULT_PH_OUT,
    DISPLACEMENT,
    TRIAL_SETUP,
    CURVATURE_SIGN,
    SEMICORE_N_OCC,
    SEMICORE_EXPECTED,
    SEMICORE_TOL,
    NKS as DEFAULT_NKS,
    K_BATCH,
    MIN_HOPPING_NORM as DEFAULT_MIN_HOPPING_NORM,
    DTYPE,
    AA_CACHE_DIR,
    _AXIS_NAMES,
    parse_ph_born_charges,
    parse_upf_zvalence,
    parse_win_atoms,
    parse_mp_grid,
    parse_win_setup,
    parse_scf_species,
    parse_scf_atoms,
    parse_scf_fermi,
    parse_centres,
    mixed_curvature_traces,
    z_electronic,
    resolve_w90_dir,
    position_source_available,
    preflight,
    centre_cross_check,
    detect_n_occ,
    check_structures,
)
from pythtb import W90
from modules import curvature as cv
from modules.axion import prepare_lambda
from wannier.wannier_io import check_mmn_compatibility, load_Lambda_R
from wannier.position_matrix import (
    apply_orbital_dependent_R_mapping,
    build_position_matrix_from_mmn,
    install_postw90_position_matrix,
)

## Config

Every flag the old CLI took is a plain variable here. `None` means "use the library default" (shown in the comment).

In [ ]:
RUN_COMPUTATION = False  # True: rebuild everything from the Wannier90 runs. False: load a previously saved born_charge/*.npz.

SELF_TEST = False               # --self-test: base model as both endpoints; Z^el must be 0
NKS = None                      # --nks; None -> DEFAULT_NKS = [4,6,8,10,12,14,16]
DISPLACED_ATOM = None           # --displaced-atom; None -> "Ti"
TRIAL_OVERRIDE = None           # --trial; None -> DISPLACEMENT[atom]["default_trial"]
N_OCC_OVERRIDE = None           # --n-occ; None -> auto-detected
POSITION_SOURCE = "mmn"         # --position-source: rdat | mmn | rfull
WB_TRANSL_INV = "none"          # --wb-transl-inv: none | mv  (only meaningful for position_source='mmn')
WB_WS_DIST_TOL = 1e-5           # --wb-ws-dist-tol
EMBEDDING = "centres"           # --embedding: centres | zero
WS_REASSIGN = True              # --ws-reassign / --no-ws-reassign
MIN_HOPPING_NORM_OVERRIDE = None  # --min-hopping-norm; None -> DEFAULT_MIN_HOPPING_NORM
SEMICORE_TEST = False           # --semicore-test

nks = NKS if NKS else DEFAULT_NKS
assert np.isfinite(WB_WS_DIST_TOL) and WB_WS_DIST_TOL > 0, "WB_WS_DIST_TOL must be positive and finite"
assert POSITION_SOURCE == "mmn" or WB_TRANSL_INV == "none", "WB_TRANSL_INV applies only to POSITION_SOURCE='mmn'"
assert POSITION_SOURCE != "rfull" or WS_REASSIGN, "POSITION_SOURCE='rfull' requires WS_REASSIGN=True"

## Resolve directories

In [ ]:
if RUN_COMPUTATION:
    displaced_atom = DISPLACED_ATOM or "Ti"
    _disp = DISPLACEMENT[displaced_atom]
    MODE_RUN = _disp["mode_run"]
    DISPLACED_UPF = BASE_RUN / _disp["upf"]
    MMN_DONOR = _disp["mmn_donor"]
    trial = TRIAL_OVERRIDE or _disp["default_trial"]
    if trial not in _disp["excluded_on_atom"]:
        raise ValueError(
            f"No semicore attribution recorded for {displaced_atom} with {trial}. "
            f"Add it to DISPLACEMENT['{displaced_atom}']['excluded_on_atom']."
        )
    print(f"displaced atom: {displaced_atom}  ({MODE_RUN.relative_to(STO)})")
    base_w90 = resolve_w90_dir(BASE_RUN / trial, "base")
    mode_root = MODE_RUN / trial / "no_displacement"
    if not mode_root.is_dir():
        # the displaced runs are not always named identically to the base ones
        # (e.g. "..._O_sp" vs "..._Osp"); match on the trial_NN prefix instead.
        stem = trial.split("_")[0] + "_" + trial.split("_")[1]
        cands = sorted(d for d in MODE_RUN.glob(f"{stem}*") if d.is_dir())
        if len(cands) == 1:
            mode_root = cands[0] / "no_displacement"
            print(f"  mode: {trial} not found, matched {cands[0].name} on the '{stem}' prefix")
        elif len(cands) > 1:
            raise ValueError(f"{stem} is ambiguous under {MODE_RUN}: {[d.name for d in cands]}")
    mode_w90 = resolve_w90_dir(mode_root, "mode")
    donor = MMN_DONOR.get(trial, {})

## Preflight

In [ ]:
if RUN_COMPUTATION:
    missing = preflight(SELF_TEST, base_w90, mode_w90, POSITION_SOURCE, donor,
                        ph_out=DEFAULT_PH_OUT, displaced_upf=DISPLACED_UPF)
    if missing:
        raise FileNotFoundError(
            "Missing required inputs:\n  " + "\n  ".join(missing) +
            f"\n\nProduce the displaced Wannier90 run's .amn with "
            f"{MODE_RUN / 'w90/generate_rigid_shift_amn.sh'} --no-displacement, "
            "run wannier90.x there, then rerun this cell. Use SELF_TEST=True to "
            "exercise the machinery on the base model alone."
        )
    out_dir = mode_w90 / "born_charge"
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"base w90 : {base_w90}")
    print(f"mode w90 : {'(self-test: base reused)' if SELF_TEST else mode_w90}")
    print(f"output   : {out_dir}")

## Consistency checks: do the two structures differ only by the intended displacement?

In [ ]:
if RUN_COMPUTATION:
    check_structures(SELF_TEST, base_w90, mode_w90, base_run=BASE_RUN, mode_run=MODE_RUN,
                     displaced_atom=displaced_atom)

## DFPT reference and the ionic term

`z_qe` is `Z*_{Ti, z\beta}` from `ph.x` (no ASR). `z_ion` folds the semicore electrons excluded by `dis_win_min` into the ionic term -- see the module docstring.

In [ ]:
if RUN_COMPUTATION:
    z_raw = parse_ph_born_charges(DEFAULT_PH_OUT, asr=False)
    ti_entries = [(n, t) for n, (s, t) in z_raw.items() if s == displaced_atom]
    if len(ti_entries) != 1:
        raise ValueError(f"Expected one {displaced_atom} in {DEFAULT_PH_OUT}, got {len(ti_entries)}")
    z_qe = ti_entries[0][1][2]  # DISPLACED_AXIS = z
    z_val = parse_upf_zvalence(DISPLACED_UPF)
    setup = TRIAL_SETUP[trial]
    N_OCC_EXPECTED = setup["n_occ"]
    N_EXCLUDED_TOTAL = setup["excluded_total"]
    N_EXCLUDED_ON_DISPLACED = _disp["excluded_on_atom"][trial]
    z_ion = z_val - N_EXCLUDED_ON_DISPLACED
    print(f"Z*_QE[{displaced_atom},zbeta] = " + "  ".join(f"{v:+.5f}" for v in z_qe))
    print(f"z_valence={z_val:+.2f}, minus {N_EXCLUDED_ON_DISPLACED} semicore -> Z^ion,eff={z_ion:+.2f}")
    print(f"electronic part must come out near {z_qe[2] - z_ion:+.5f}")

## Load Wannier90 models

In [ ]:
if RUN_COMPUTATION:
    w90_base = W90(str(base_w90), PREFIX)
    w90_mode = w90_base if SELF_TEST else W90(str(mode_w90), PREFIX)
    if not SELF_TEST and w90_base.num_wan != w90_mode.num_wan:
        raise ValueError(f"num_wann differs: base {w90_base.num_wan}, mode {w90_mode.num_wan}")
    if not SELF_TEST and not np.allclose(w90_base.lat, w90_mode.lat, atol=1e-8):
        raise ValueError("Lattice vectors differ between the two runs.")

## Build models with the shared embedding

The embedding `tau` must be chosen before the models are built (`W90.model(orb_vecs=...)`), since `model.lattice.orb_vecs = tau` afterwards mutates a throwaway copy and is silently discarded.

In [ ]:
if RUN_COMPUTATION:
    mhn = DEFAULT_MIN_HOPPING_NORM if MIN_HOPPING_NORM_OVERRIDE is None else MIN_HOPPING_NORM_OVERRIDE
    if WS_REASSIGN:
        mp_grid = parse_mp_grid(base_w90 / f"{PREFIX}.win")
        print(f"Wigner-Seitz convention: pair-dependent, mp_grid = {mp_grid}")
        for label, w90 in (("base", w90_base), *(() if SELF_TEST else (("mode", w90_mode),))):
            apply_orbital_dependent_R_mapping(w90, mp_grid)
    else:
        print("Wigner-Seitz convention: Wannier90's min |R| (WS_REASSIGN=False)")

    _probe = w90_base.model(zero_energy=0.0, min_hopping_norm=mhn)
    nominal_orb_vecs = np.zeros_like(_probe.orb_vecs) if EMBEDDING == "zero" else _probe.orb_vecs.copy()
    del _probe

    model_base = w90_base.model(zero_energy=0.0, min_hopping_norm=mhn, orb_vecs=nominal_orb_vecs)
    model_mode = (model_base if SELF_TEST else
                  w90_mode.model(zero_energy=0.0, min_hopping_norm=mhn, orb_vecs=nominal_orb_vecs))
    for _lbl, _m in (("base", model_base), ("mode", model_mode)):
        if not np.allclose(_m.orb_vecs, nominal_orb_vecs, atol=1e-12):
            raise RuntimeError(f"{_lbl} model did not take the shared embedding.")
    n_orb = model_base.norb
    print(f"num_wann = {n_orb}")

## Install the position matrix

Branches on `POSITION_SOURCE`: `mmn` rebuilds `<0s|r|Rt>` from the `.mmn` overlaps in-house; `rfull` installs postw90's pair-weighted `_r_full.dat`; `rdat` uses whatever `W90()` already loaded from `prefix_tb.dat`/`prefix_r.dat`.

In [ ]:
if RUN_COMPUTATION and POSITION_SOURCE == "mmn":
    wb_mv = WB_TRANSL_INV == "mv"
    AA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    for label, mdl, w90dir in (("base", model_base, base_w90), ("mode", model_mode, mode_w90)):
        if label == "mode" and SELF_TEST:
            continue
        src = donor.get(label) or w90dir
        check_mmn_compatibility(w90dir, src, PREFIX)
        if src != w90dir:
            print(f"  {label}: borrowing {PREFIX}.mmn from {src.name}")
        ws_tag = f"{WB_WS_DIST_TOL:g}".replace("-", "m").replace(".", "p")
        tag_ = f"{trial}_{label}_{w90dir.name}_wb{WB_TRANSL_INV}_ws{ws_tag}"
        info = build_position_matrix_from_mmn(
            mdl, w90dir, PREFIX, mmn_directory=src, cache=AA_CACHE_DIR / f"AA_{tag_}.npz",
            translational_invariant_JM=False, translational_invariant_MV=wb_mv,
            R_distance_tolerance=WB_WS_DIST_TOL,
        )
        print(f"  {label}: A(R) rebuilt from .mmn ({len(mdl._pos_r)} R-vectors); "
              f"pair Hermiticity rel {info['mmn_pair_hermiticity_relative']:.2e}")

In [ ]:
if RUN_COMPUTATION and POSITION_SOURCE == "rfull":
    for label, mdl, w90, w90dir in (("base", model_base, w90_base, base_w90),
                                    ("mode", model_mode, w90_mode, mode_w90)):
        if label == "mode" and SELF_TEST:
            continue
        centers_cartesian = np.asarray(w90.lattice.orb_vecs, dtype=float) @ np.asarray(w90.lat, dtype=float)
        info = install_postw90_position_matrix(mdl, w90dir / f"{PREFIX}_r_full.dat",
                                               centers_cartesian=centers_cartesian)
        print(f"  {label}: loaded {info['n_R_pos']} A(R) blocks; "
              f"Hermiticity rel {info['pair_hermiticity_relative']:.2e}; "
              f"centres error {info['centres_max_error']:.2e} A")

In [ ]:
if RUN_COMPUTATION:
    if model_base._pos_r is None or model_mode._pos_r is None:
        raise ValueError("The selected position-matrix source was not installed.")
    print(f"n_R(pos) = {len(model_base._pos_r)}")

## Align H(R) and A(R) to the same R-set

The pair-dependent reassignment can prune H(R) blocks that are all-zero while A(R) is still nonzero there; restore explicit zero H blocks so the two operators share one R-set (no Bloch Hamiltonian changes).

In [ ]:
if RUN_COMPUTATION:
    zero_h_blocks_added = []
    for label, mdl in (("base", model_base), ("mode", model_mode)):
        h_set = set(mdl._ham_r)
        a_set = set(mdl._pos_r)
        if h_set - a_set:
            raise ValueError(f"{label}: {len(h_set-a_set)} H(R) vectors have no A(R) block.")
        missing_h = a_set - h_set
        if missing_h:
            sample_h = next(iter(mdl._ham_r.values()))
            for R in missing_h:
                mdl._ham_r[R] = np.zeros_like(sample_h)
                mdl._ham_deg[R] = 1.0
            print(f"  {label}: restored {len(missing_h)} explicit zero H(R) blocks")
        if set(mdl._ham_r) != set(mdl._pos_r):
            raise RuntimeError(f"{label}: failed to align H(R) and A(R) sets")
        zero_h_blocks_added.append(len(missing_h))
    print(f"verified identical H/A R sets: base={len(model_base._ham_r)}, mode={len(model_mode._ham_r)}")

## Occupancy and electron accounting

In [ ]:
if RUN_COMPUTATION:
    e_ref = parse_scf_fermi(BASE_RUN / f"{PREFIX}.scf.out")
    n_occ = detect_n_occ(model_base, e_ref)
    forced = N_OCC_OVERRIDE
    if forced is not None and n_occ != forced:
        print(f"  !! overriding detected occupancy {n_occ} with {forced}")
        n_occ = forced
    if n_occ > n_orb:
        raise ValueError(f"n_occ = {n_occ} exceeds the {n_orb}-Wannier space.")
    all_occupied = n_occ == n_orb
    if all_occupied:
        print("  occupied-only Wannierization: B_int and B_cross vanish identically")

    zval = {s: parse_upf_zvalence(BASE_RUN / upf) for s, upf in parse_scf_species(BASE_RUN / f"{PREFIX}.scf.in")}
    species, _ = parse_scf_atoms(BASE_RUN / f"{PREFIX}.scf.in")
    n_elec = sum(zval[s] for s in species)
    n_excluded = int(round(n_elec)) - n_occ
    print(f"  occupied Wannier states = {n_occ}  ->  {n_excluded} occupied band(s) outside the window")
    if n_excluded == 0:
        n_excl_atom = 0
    elif n_excluded != N_EXCLUDED_TOTAL:
        raise ValueError(f"{n_excluded} bands outside the window but N_EXCLUDED_TOTAL={N_EXCLUDED_TOTAL}")
    else:
        n_excl_atom = N_EXCLUDED_ON_DISPLACED
        print(f"  of those, {n_excl_atom} sit on {displaced_atom} (ionic term)")

## External fields: `position_terms` with the fixed embedding

In [ ]:
if RUN_COMPUTATION:
    lambda_file = mode_w90 / f"{PREFIX}_lambda_u.npz"
    lambda_r = load_Lambda_R(lambda_file) if lambda_file.exists() else None
    lambda_prepared = None if lambda_r is None else prepare_lambda(lambda_r, DTYPE)
    # tau = model.orb_vecs = nominal_orb_vecs for both structures (asserted where they are built)
    pos_base = cv.position_terms(model_base, dtype=DTYPE)
    pos_mode = pos_base if SELF_TEST else cv.position_terms(model_mode, dtype=DTYPE)
    print("frozen Wannier gauge (A_u = 0)" if lambda_r is None else f"Lambda(R) loaded from {lambda_file.name}")

    if SEMICORE_TEST:
        semicore_n_occ = _disp["semicore_n_occ"].get(trial, SEMICORE_N_OCC)
        if semicore_n_occ is None:
            raise ValueError(f"SEMICORE_TEST has no calibrated null for {trial}.")
        n_occ = semicore_n_occ
        print(f"SEMICORE_TEST: integrating states 0..{n_occ - 1} only; "
              f"Z^el_zz must come out near {SEMICORE_EXPECTED:+.0f}")
    occ = np.arange(n_occ, dtype=int)
    B_recip = np.asarray(model_base.recip_lat_vecs, dtype=float)
    B_inv = np.linalg.inv(B_recip)

## Wannier-centre cross-check

Exact when `n_occ == num_wann` (occupied-only Wannierization): `Z^el = -d/du sum_s r_bar_s` from `prefix_centres.xyz` alone, no curvature, no k-mesh.

In [ ]:
if RUN_COMPUTATION:
    z_centres = None
    if n_occ == n_orb and not SELF_TEST:
        z_centres = centre_cross_check(base_w90, mode_w90, 0.01)
        if z_centres is not None:
            print("Wannier-centre cross-check: Z^el = " + "  ".join(f"{v:+.5f}" for v in z_centres))
    Omega = abs(np.linalg.det(np.asarray(model_base.lattice.lat_vecs, dtype=float)))
    print(f"cell volume = {Omega:.4f} A^3")

## BZ mesh sweep

For each `nk`, accumulate `Tr B_int`, `Tr B_cross`, `Tr B_ext` over the whole mesh in batches of `K_BATCH`, then convert to Cartesian `Z^el`. `last_batch_traces` after this cell is the raw per-k, per-piece curvature trace from the *last* nk's *last* batch -- inspect it directly if you want the actual curvature values rather than just the BZ-averaged `Z*`.

In [ ]:
if RUN_COMPUTATION:
    rows = []
    last_batch_traces = None
    for nk in nks:
        k_mesh = model_base.k_uniform_mesh([nk, nk, nk], include_endpoints=False)
        n_k = k_mesh.shape[0]
        acc = {key: np.zeros(3, dtype=complex) for key in ("int", "cross", "ext")}
        gap = np.inf
        print(f"=== nk = {nk}  ({n_k} k-points) ===")
        for i in range(0, n_k, K_BATCH):
            j = min(i + K_BATCH, n_k)
            kb = k_mesh[i:j]
            e = np.linalg.eigvalsh(model_base.hamiltonian(kb, flatten_spin_axis=True))
            if n_occ < n_orb:
                gap = min(gap, float(np.min(e[:, n_occ] - e[:, n_occ - 1])))
            tr = mixed_curvature_traces(model_base, model_mode, kb, n_occ, 0.01, pos_base, pos_mode, lambda_prepared)
            last_batch_traces = tr
            for key, val in tr.items():
                acc[key] += val.sum(axis=1)
        z_parts = z_electronic(acc, n_k, B_inv)
        z_el = sum(z_parts.values())
        z_tot = z_el.real.copy()
        z_tot[2] += z_ion
        resid = z_tot - z_qe
        print(f"  Z*,Wannier = " + "  ".join(f"{v:+.5f}" for v in z_tot))
        print(f"  Z*,QE      = " + "  ".join(f"{v:+.5f}" for v in z_qe))
        rows.append({"nk": nk, "n_k": n_k, "gap": gap, "z_int": z_parts["int"].real,
                     "z_cross": z_parts["cross"].real, "z_ext": z_parts["ext"].real,
                     "z_el": z_el.real, "z_tot": z_tot, "z_qe": z_qe, "resid": resid})
    last_batch_traces["int"].shape  # (3, n_k_in_last_batch); same for "cross", "ext"

## Save

In [ ]:
if RUN_COMPUTATION:
    tag = ("self_test" if SELF_TEST else "semicore" if SEMICORE_TEST else MODE_RUN.relative_to(STO).parts[0])
    if POSITION_SOURCE != "rdat" or EMBEDDING != "centres":
        tag += f"_{POSITION_SOURCE}_tau{EMBEDDING}"
    if mhn != DEFAULT_MIN_HOPPING_NORM:
        tag += f"_mhn{mhn:g}"
    if not WS_REASSIGN:
        tag += "_nows"
    if POSITION_SOURCE == "mmn":
        tag += f"_wb{WB_TRANSL_INV}"
    npz_path = out_dir / f"born_charge_{tag}.npz"
    np.savez_compressed(
        npz_path,
        nks=np.array([r["nk"] for r in rows], dtype=np.int32),
        n_k=np.array([r["n_k"] for r in rows], dtype=np.int64),
        min_gap=np.array([r["gap"] for r in rows]),
        z_int=np.array([r["z_int"] for r in rows]), z_cross=np.array([r["z_cross"] for r in rows]),
        z_ext=np.array([r["z_ext"] for r in rows]), z_el=np.array([r["z_el"] for r in rows]),
        z_total=np.array([r["z_tot"] for r in rows]), z_qe=z_qe, z_ion=np.float64(z_ion),
        d_beta=np.float64(0.01), n_occ=np.int32(n_occ), num_wann=np.int32(n_orb),
        z_valence=np.float64(z_val), position_source=np.array([POSITION_SOURCE]),
        embedding=np.array([EMBEDDING]), ws_reassign=np.int8(WS_REASSIGN),
        min_hopping_norm=np.float64(mhn), trial=np.array([trial]),
    )
    csv_path = out_dir / f"born_charge_{tag}.csv"
    with csv_path.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["nk", "n_k", "min_gap_eV"]
                   + [f"z_{p}_{a}" for p in ("int", "cross", "ext", "el", "total") for a in _AXIS_NAMES]
                   + [f"z_qe_{a}" for a in _AXIS_NAMES])
        for r in rows:
            w.writerow([r["nk"], r["n_k"], f"{r['gap']:.6f}"]
                       + [f"{v:.8f}" for p in ("z_int", "z_cross", "z_ext", "z_el", "z_tot") for v in r[p]]
                       + [f"{v:.8f}" for v in r["z_qe"]])
    print(f"saved {npz_path}")
    print(f"saved {csv_path}")

## Load a previous run instead (skip the cells above)

Defaults to the canonical trial_03 Ti result (position_source=mmn, embedding=centres, ws_reassign=True). Point `LOAD_NPZ` at a different saved `born_charge_*.npz` to load another trial/flag combination -- it lives next to the Wannier90 run that produced it, under `<mode_w90>/born_charge/`, not under `validation/results/` (see the module docstring).

In [ ]:
if not RUN_COMPUTATION:
    from validation._paths import STO_TI_Q_01A_RUN
    LOAD_NPZ = (DATA / "SrTiO3/Ti_Q_01A/output/181839bc889/trial_03_Sr_sp_Ti_pd_Osp/no_displacement/185119bc889"
                / "born_charge/born_charge_Ti_Q_01A_mmn_taucentres_wbnone.npz")  # set to None, or another born_charge_*.npz, to change what's loaded
    if LOAD_NPZ is not None and LOAD_NPZ.exists():
        with np.load(LOAD_NPZ, allow_pickle=False) as data:
            loaded = {k: data[k] for k in data.files}
        rows = [
            {"nk": int(loaded["nks"][i]), "z_int": loaded["z_int"][i], "z_cross": loaded["z_cross"][i],
             "z_ext": loaded["z_ext"][i], "z_el": loaded["z_el"][i], "z_tot": loaded["z_total"][i]}
            for i in range(len(loaded["nks"]))
        ]
        z_qe = loaded["z_qe"]
        print(f"loaded {LOAD_NPZ}")
    else:
        print("RUN_COMPUTATION is False and LOAD_NPZ is None -- set LOAD_NPZ to a saved born_charge_*.npz "
              "path, or set RUN_COMPUTATION = True above.")
        rows = None

## Benchmark table + convergence plot

In [ ]:
if rows:
    print(f"{'nk':>4}  {'Wannier':>10}  {'QE (DFPT)':>10}  {'diff':>9}  {'rel':>7}  {'zx':>9}  {'zy':>9}")
    for r in rows:
        resid = r["z_tot"][2] - z_qe[2]
        rel = 100.0 * resid / z_qe[2]
        print(f"{r['nk']:>4}  {r['z_tot'][2]:>10.5f}  {z_qe[2]:>10.5f}  {resid:>+9.5f}  {rel:>+6.2f}%  "
              f"{r['z_tot'][0]:>+9.5f}  {r['z_tot'][1]:>+9.5f}")

    fig, (ax_z, ax_parts) = plt.subplots(1, 2, figsize=(11, 4.2), dpi=150)
    nk_vals = [r["nk"] for r in rows]
    ax_z.axhline(float(z_qe[2]), color="0.3", linestyle="--", linewidth=1.2, label="DFPT")
    ax_z.plot(nk_vals, [r["z_tot"][2] for r in rows], "o-", color="C0", label="Wannier total")
    ax_z.set_xlabel("nk"); ax_z.set_ylabel("$Z^*_{Ti,zz}$"); ax_z.legend(frameon=False); ax_z.grid(alpha=0.2)
    for key, color in (("z_int", "C1"), ("z_cross", "C2"), ("z_ext", "C3")):
        ax_parts.plot(nk_vals, [r[key][2] for r in rows], "o-", color=color, label=key)
    ax_parts.set_xlabel("nk"); ax_parts.set_ylabel("$Z^{el}_{zz}$ by piece"); ax_parts.legend(frameon=False); ax_parts.grid(alpha=0.2)
    fig.suptitle("Ti Born effective charge: DFPT vs Wannier mixed curvature")
    fig.tight_layout()
    plt.show()
else:
    print("nothing to plot -- run the computation or set LOAD_NPZ above.")